# 06 — Kết hợp các kỹ thuật: bổ sung hay trùng nhau?

[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/vinumqa-numerical-reasoning/blob/main/notebooks/06_combination.ipynb)

**Cần GPU.** ~2 giờ nếu đã chạy xong nấc 1–5 (chỉ chạy các ô còn thiếu).

## Vì sao cần notebook này

Năm nấc trước là một **chuỗi**: mỗi nấc thêm một kỹ thuật lên trên nấc trước. Nhưng chuỗi
không trả lời được câu hỏi quan trọng nhất:

> **SFT và ACE đều học từ tập train** — một cái nhét vào trọng số, một cái nhét vào ngữ cảnh.
> Chúng bổ sung cho nhau, hay đang học cùng một thứ?

Tương tự với self-eval và ACE: cả hai đều là **cơ chế sửa lỗi lúc suy luận**, nên rất có thể
ACE chỉ sửa đúng những lỗi self-eval vốn đã sửa được.

Chuỗi tuyến tính không tách được điều này. Phải chạy **ma trận đầy đủ**.

## Ma trận

Ba yếu tố nhị phân, đều trên nền prompt có cấu trúc (nấc 2):

| Ô | SFT | Self-eval | ACE | Trùng với nấc |
|---|:---:|:---:|:---:|---|
| `E` | ❌ | ❌ | ❌ | nấc 2 |
| `E+A` | ❌ | ❌ | ✅ | — *(mới)* |
| `E+S` | ❌ | ✅ | ❌ | nấc 4 |
| `E+S+A` | ❌ | ✅ | ✅ | nấc 5 |
| `F` | ✅ | ❌ | ❌ | nấc 3 |
| `F+A` | ✅ | ❌ | ✅ | — *(mới)* |
| `F+S` | ✅ | ✅ | ❌ | nấc 4b |
| `F+S+A` | ✅ | ✅ | ✅ | nấc 5b — **tổ hợp đầy đủ** |

Ô nào đã chạy ở nấc trước thì **đọc lại từ đĩa**, không chạy lại. Notebook chỉ sinh các ô còn
thiếu.

## Notebook này trả lời

1. **Tác động chính** của từng kỹ thuật — trung bình trên mọi thiết lập của hai kỹ thuật kia.
2. **Tương tác** — ACE có giúp nhiều hơn hay ít hơn khi model đã được SFT?
3. **Tổ hợp tốt nhất**, và nó có thật sự hơn cấu hình tham chiếu không.
4. **Chi phí trên mỗi điểm EA** — kỹ thuật nào đáng đồng tiền nhất.

## §1. Môi trường

In [ ]:
# Cài đặt — ghim theo bộ ĐÃ XÁC MINH cài xong sạch trên image Colab hiện tại
# (Python 3.13, torch 2.11.0+cu128, A100).
#
# ⚠ KHÁC bản tham chiếu, và đây là chủ ý:
#   Khối cài đặt gốc ghim transformers==4.56.2 / trl==0.22.2 / xformers==0.0.29.post3.
#   Trên image Colab hiện tại nó THẤT BẠI — nhánh chọn xformers chỉ biết torch 2.8/2.9,
#   gặp torch 2.11 thì rơi vào bản 0.0.29.post3 (dành cho torch 2.5) nên đổ cả khối,
#   mà `%%capture` lại nuốt mất báo lỗi.
#   Bộ dưới đây là bộ pip tự giải ra khi để `unsloth` và `vllm` thoả thuận với nhau.
#   Chênh lệch phiên bản được ghi vào `env` của meta mỗi nấc, nên báo cáo vẫn truy được.
#
# ⏱ 6–12 phút (đã ghim nên pip khỏi dò tìm). Cố ý KHÔNG giấu output để thấy nó còn sống.
import os, time
_t_cai = time.time()
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install unsloth==2026.9.4 transformers==4.57.6 trl==0.24.0 peft==0.20.0 bitsandbytes==0.50.2 xformers==0.0.35
    # vLLM phải khớp CUDA của torch. Bản trên PyPI dựng cho CUDA 13, còn Colab đang
    # CUDA 12.8 → unsloth CHẶN import và báo "No module named 'vllm'" dù gói vẫn có.
    # Wheel dưới đây là bản cu129, đúng cái unsloth khuyến nghị cho hệ CUDA 12.x.
    # Nếu image Colab đổi CUDA: chạy ô này, đọc dòng WARNING của unsloth ở cell sau —
    # nó in ra đúng URL wheel cần dùng, thay vào đây là xong.
    !pip install https://github.com/vllm-project/vllm/releases/download/v0.23.0/vllm-0.23.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl
print(f"\n[CÀI ĐẶT] xong sau {(time.time() - _t_cai) / 60:.1f} phút")
print("[CÀI ĐẶT] Colab hiện nút RESTART SESSION thì bấm, rồi chạy lại TỪ CELL #2 "
      "(bỏ qua ô này — cài lại là thừa).")

# Riêng nấc 5 (ACE) cần thêm truy hồi ngữ nghĩa đa ngữ + từ vựng.
# ⚠ Phải GHIM 5.7.0: từ 6.0 trở đi sentence-transformers đòi transformers>=5, mà
#   vLLM 0.23 lại cấm toàn bộ transformers 5.x. Lệnh này chạy sau cùng nên không ghim
#   là nó nâng transformers lên 5.x và làm hỏng vLLM vừa cài xong.
!pip install -q "sentence-transformers==5.7.0" "rank-bm25==0.2.2"

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CẤU HÌNH — CHỈ SỬA MỘT DÒNG, MỘT LẦN, Ở NOTEBOOK 00                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Code + dữ liệu lấy thẳng từ GitHub: cell này tự clone lần đầu và tự cập nhật
# những lần sau, nên sửa code dưới máy chỉ cần `git push` là Colab có bản mới.
# Riêng KẾT QUẢ ghi lên Drive để không mất khi Colab ngắt session.
GITHUB_REPO = "https://github.com/ThanhDatVN/vinumqa-numerical-reasoning"
OUTPUT_DIR  = "/content/drive/MyDrive/vinumqa_runs"

# Hai dòng dưới để trống là được — chỉ điền khi muốn tự quyết:
#   REPO_DIR  chỗ đã có sẵn code, điền vào thì bỏ qua bước clone
#   DATA_DIR  chỗ để dữ liệu, nếu tách khỏi code
REPO_DIR = ""
DATA_DIR = ""
# ──────────────────────────────────────────────────────────────────────────────

import os, sys, json, time, csv, gc, random, glob, shutil, subprocess
from collections import Counter, defaultdict
from datetime import datetime
import numpy as np

_PLACEHOLDER = "TEN-TAI-KHOAN"


def _is_repo(p):
    """Thư mục p có phải bản sao của dự án không."""
    return bool(p) and os.path.isdir(os.path.join(p, "vinumqa"))


# Điền sẵn REPO_DIR = đã tự lo chỗ để code, cell này không đụng gì tới git.
_pinned = bool(str(REPO_DIR).strip())

ON_COLAB  = "COLAB_" in "".join(os.environ.keys())
_MEMO = ("/content/drive/MyDrive/.vinumqa_paths.json" if ON_COLAB
         else os.path.join(os.path.expanduser("~"), ".vinumqa_paths.json"))

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
else:                                  # chạy dưới máy: repo là thư mục đang đứng, hoặc cha nó
    _here = os.path.abspath(os.getcwd())
    for _c in (_here, os.path.dirname(_here), os.path.dirname(os.path.dirname(_here))):
        if _is_repo(_c):
            REPO_DIR, OUTPUT_DIR, _pinned = _c, os.path.join(_c, "runs"), True; break

# ─── Ghi nhớ cấu hình: GITHUB_REPO chỉ phải điền một lần, ở notebook 00 ───
_saved = {}
if os.path.exists(_MEMO):
    try:
        _saved = json.load(open(_MEMO, encoding="utf-8"))
    except Exception:
        _saved = {}
if _PLACEHOLDER in GITHUB_REPO and _saved.get("GITHUB_REPO"):
    GITHUB_REPO = _saved["GITHUB_REPO"]
    print("[CẤU HÌNH] dùng GITHUB_REPO đã ghi nhớ từ lần chạy trước")
DATA_DIR = DATA_DIR or _saved.get("DATA_DIR", "")

# ─── Lấy code + dữ liệu về ───
if _pinned:                                      # code đã có sẵn, không clone
    if not _is_repo(REPO_DIR):
        raise FileNotFoundError(
            f"Không thấy package tại {REPO_DIR}/vinumqa.\n"
            f"REPO_DIR phải trỏ tới thư mục chứa vinumqa/, data/, notebooks/ — "
            f"hoặc để trống REPO_DIR để tự clone từ GITHUB_REPO.")
    print(f"[CODE] {REPO_DIR} (chỉ định sẵn)")
else:
    if _PLACEHOLDER in GITHUB_REPO:
        raise ValueError(
            "Chưa điền GITHUB_REPO ở ĐẦU CELL NÀY.\n\n"
            "Sửa thành URL repo của bạn, ví dụ:\n"
            "    GITHUB_REPO = \"https://github.com/ten-cua-ban/vinumqa-ladder\"\n\n"
            "Chỉ cần sửa MỘT LẦN ở notebook 00 — bảy notebook sau tự đọc lại.")
    _url  = GITHUB_REPO.strip().rstrip("/")
    _url  = _url if _url.endswith(".git") else _url + ".git"
    REPO_DIR = os.path.join("/content" if ON_COLAB else os.getcwd(),
                            os.path.basename(_url)[:-len(".git")])
    if _is_repo(REPO_DIR):        # còn lại sau khi restart runtime → lấy bản mới nhất
        _g = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"],
                            capture_output=True, text=True)
        print("[CODE] " + REPO_DIR + " — " +
              ("đã cập nhật bản mới nhất" if _g.returncode == 0 else "giữ bản đang có"))
    else:
        if os.path.exists(REPO_DIR) and os.listdir(REPO_DIR) \
                and not os.path.isdir(os.path.join(REPO_DIR, ".git")):
            raise RuntimeError(
                f"{REPO_DIR} đã tồn tại và không phải bản clone của dự án.\n"
                f"Xoá nó, hoặc điền REPO_DIR ở đầu cell này cho trỏ đúng chỗ có code.")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        print(f"[CODE] đang clone {_url} … (~25 MB, khoảng 15 giây)")
        _g = subprocess.run(["git", "clone", "--depth", "1", _url, REPO_DIR],
                            capture_output=True, text=True)
        if _g.returncode or not _is_repo(REPO_DIR):
            raise RuntimeError(
                "git clone thất bại:\n" + (_g.stderr or "")[-800:] + "\n\n"
                "Kiểm tra lại URL. Nếu repo để private thì dùng dạng có token:\n"
                "    https://<token>@github.com/<tài-khoản>/<repo>")
        print(f"[CODE] → {REPO_DIR}")

sys.path.insert(0, REPO_DIR)

try:                                   # ghi nhớ cho các notebook sau
    json.dump({"GITHUB_REPO": GITHUB_REPO, "REPO_DIR": REPO_DIR,
               "OUTPUT_DIR": OUTPUT_DIR, "DATA_DIR": DATA_DIR},
              open(_MEMO, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
except Exception:
    pass

from vinumqa import data, dsl, io_utils, pipeline, sft, stats
from vinumqa.ace import playbook as pb_mod
from vinumqa.prompts import PromptKit

# ─── Bố cục thư mục làm việc ───
DATA_DIR    = DATA_DIR or os.path.join(REPO_DIR, "data")
if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f"Không thấy dữ liệu tại {DATA_DIR} (cần train.json / valid.json / test.json).\n"
        f"Dữ liệu nằm trong repo, nên thường là do repo thiếu thư mục data/ "
        f"— kiểm tra đã push data/ lên GitHub chưa, hoặc điền DATA_DIR ở đầu cell này.")
RESULT_DIR  = os.path.join(OUTPUT_DIR, "stages")      # kết quả từng nấc (dùng chung)
LOG_DIR     = os.path.join(OUTPUT_DIR, "logs")        # output thô của model
ARTIFACT_DIR= os.path.join(OUTPUT_DIR, "artifacts")   # playbook, adapter, biểu đồ
for _d in (OUTPUT_DIR, RESULT_DIR, LOG_DIR, ARTIFACT_DIR):
    os.makedirs(_d, exist_ok=True)

STAMP = datetime.now().strftime("%Y%m%d_%H%M")
RANDOM_SEED = 42
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

splits = data.load_all(DATA_DIR)
train_all = [s for s in splits["train"] if data.has_gold(s)]
valid_all = [s for s in splits["valid"] if data.has_gold(s)]
test_all  = splits["test"]


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ĐỌC / GHI KẾT QUẢ CÁC NẤC                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# Mọi notebook ghi kết quả vào RESULT_DIR theo cùng một quy ước, nên notebook
# sau đọc lại được của notebook trước mà không phải chỉnh đường dẫn.

# Thang prompt LỒNG NHAU: basic ⊂ no_fewshot ⊂ engineered — mỗi nấc thêm đúng một khối
# cắt ra từ prompt hoàn chỉnh, phần chung giống nhau từng ký tự.
LADDER = [
    ("01_basic",           "Nấc 1 — prompt cơ bản (danh sách phép toán + yêu cầu)"),
    ("02_prompt_eng",      "Nấc 2 — prompt hoàn chỉnh (+ hướng dẫn từ khoá + few-shot)"),
    ("03_sft",             "Nấc 3 — + SFT Qwen3-8B"),
    ("04_selfeval_base",   "Nấc 4 — + self-eval (model gốc)"),
    ("05_ace_base",        "Nấc 5 — + ACE (model gốc)"),
    ("05c_ace_basic_base", "Nấc 5c — ACE trên prompt cơ bản"),
    ("05_ace_random_base", "Đối chứng — bullet ngẫu nhiên"),
    ("05c_ace_basic_random_base", "Đối chứng — bullet ngẫu nhiên, prompt cơ bản"),
    ("06_comb_E_A",        "Tổ hợp — prompt + ACE (không self-eval)"),
    ("08_tu_nhat_quan",    "Mới — self-consistency K mẫu (ví dụ cố định)"),
    ("09_vidu_dong",       "Mới — self-consistency + ví dụ truy hồi"),
    ("04_selfeval_base_moi", "Mục tiêu — self-eval + K mẫu + ví dụ truy hồi"),
    ("10_bo_chon",         "Mới — model tự chấm giữa các ứng viên"),
]
LADDER_LABEL = dict(LADDER)

# Những biến phải đi kèm kết quả thì mới truy lại được về sau.
_CFG_KEYS = ("MODEL_NAME", "MODEL_TAG", "TEMPERATURE", "MAX_TOKENS", "REPETITION_PENALTY",
             "MAX_SEQ_LENGTH", "BATCH_SIZE", "GPU_MEM_UTIL", "MAX_NUM_SEQS", "RANDOM_SEED")


def run_env():
    """Môi trường THẬT lúc chạy: commit, GPU, phiên bản thư viện.

    Chỉ đọc thư viện đã nạp (``sys.modules``) chứ không import thêm — vừa nhanh,
    vừa báo đúng những gì thật sự được dùng.
    """
    env = {"python": sys.version.split()[0]}
    try:                                   # bản code nào sinh ra kết quả này
        def _g(*a):
            return subprocess.run(["git", "-C", REPO_DIR, *a],
                                  capture_output=True, text=True).stdout.strip()
        env["commit"] = _g("rev-parse", "--short", "HEAD")
        env["branch"] = _g("rev-parse", "--abbrev-ref", "HEAD")
        env["dirty"] = bool(_g("status", "--porcelain"))
    except Exception:
        pass
    _torch = sys.modules.get("torch")
    if _torch is not None:
        env["torch"] = getattr(_torch, "__version__", "?")
        try:
            if _torch.cuda.is_available():
                _p = _torch.cuda.get_device_properties(0)
                env["gpu"] = _p.name
                env["vram_gb"] = round(_p.total_memory / 1024**3, 1)
                env["cc"] = f"{_p.major}.{_p.minor}"
            else:
                env["gpu"] = "CPU"
        except Exception:
            pass
    else:
        env["gpu"] = "CPU (không nạp torch)"
    for _lib in ("transformers", "trl", "peft", "vllm", "unsloth",
                 "sentence_transformers", "numpy"):
        _m = sys.modules.get(_lib)
        if _m is not None and hasattr(_m, "__version__"):
            env[_lib] = _m.__version__
    return env


def stage_path(stage, kind="jsonl"):
    """Đường dẫn chuẩn của một nấc. kind ∈ {jsonl, meta}."""
    return os.path.join(RESULT_DIR, {
        "jsonl": f"{stage}.jsonl",
        "meta":  f"{stage}_meta.json"}[kind])


# Cấu hình CHUẨN của cả thang bậc — đo trên A100 40GB.
# MAX_SEQ_LENGTH đổi theo GPU (A100 15000 / L4 13500 / T4 8192), mà đổi GPU là đổi
# thành phần lô, đổi kernel, đổi luôn token được lấy mẫu ở temperature 0.1. Hai lần
# chạy khác max_seq KHÔNG so thẳng được, nên phải ghi sang tên nấc khác.
MAX_TOKENS_CHUAN, MAX_SEQ_CHUAN = 4096, 17000



def save_stage(stage, rows, metrics, extra=None, quiet=False):
    """Ghi kết quả một nấc: jsonl + meta, kèm một file output thô.

    Chạy với ``MAX_TOKENS`` khác mức chuẩn thì tự ghi sang tên nấc khác. Đổi trần sinh
    là đổi cấu hình, kết quả không so thẳng với thang bậc được — mà nếu cứ ghi đè lên
    tên cũ thì mất luôn bản chuẩn, không lấy lại được nếu không chạy lại GPU.
    """
    _hau_to = ""
    _mt, _ms = globals().get("MAX_TOKENS"), globals().get("MAX_SEQ_LENGTH")
    if _mt and _mt != MAX_TOKENS_CHUAN:
        _hau_to += f"_tok{_mt}"
    if _ms and _ms != MAX_SEQ_CHUAN:
        _hau_to += f"_seq{_ms}"
    if _hau_to and not stage.endswith(_hau_to):
        stage = f"{stage}{_hau_to}"
        if not quiet:
            print(f"[GHI] ⚠ cấu hình khác chuẩn (max_tokens={_mt}, max_seq={_ms}, "
                  f"GPU={globals().get('_GPU', '?')}) → ghi sang nấc '{stage}'.")
            print( "       Kết quả khác GPU/khác trần KHÔNG so thẳng với thang bậc chuẩn.")
    io_utils.save_full_jsonl(rows, stage_path(stage, "jsonl"))
    io_utils.save_raw_jsonl(rows, os.path.join(LOG_DIR, f"{stage}_raw_{STAMP}.jsonl"))
    meta = {"stage": stage, "label": LADDER_LABEL.get(stage, stage),
            "stamp": STAMP, "n": len(rows), "metrics": metrics,
            "model": globals().get("MODEL_NAME"),
            "temperature": globals().get("TEMPERATURE"),
            "max_tokens": globals().get("MAX_TOKENS"),
            "max_seq_length": globals().get("MAX_SEQ_LENGTH"),
            "ctx_truncated": bool(getattr(globals().get("prompt_kit", None),
                                          "max_ctx_chars", None)),
            "enable_thinking": getattr(globals().get("prompt_kit", None),
                                       "enable_thinking", "?"),
            "ty_le_bi_cat_token": (round(ty_le_bi_cat(), 4)
                                   if "ty_le_bi_cat" in globals() else None),
            "bi_cat_theo_buoc": (bi_cat_theo_buoc()
                                if "bi_cat_theo_buoc" in globals() else None),
            "nap_an_toan": bool(globals().get("NAP_AN_TOAN", False)),
            "config": {k: globals()[k] for k in _CFG_KEYS if k in globals()},
            "env": run_env(),
            **(extra or {})}
    with open(stage_path(stage, "meta"), "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=1, default=str)
    if not quiet:
        print(f"[GHI] nấc '{stage}':")
        print(f"      {stage_path(stage, 'jsonl')}   ← notebook sau đọc file này")
        print(f"      {stage_path(stage, 'meta')}")
    return stage                      # tên THẬT, có thể khác tên truyền vào


def load_stage(stage, quiet=False):
    """Đọc lại kết quả một nấc, đã sắp đúng thứ tự test_all. None nếu chưa có."""
    p = stage_path(stage, "jsonl")
    if not os.path.exists(p):
        if not quiet:
            print(f"[ĐỌC] ⚠ chưa có '{stage}' — chạy notebook tương ứng trước.")
        return None
    rows = [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]
    order = {s["id"]: i for i, s in enumerate(test_all)}
    rows.sort(key=lambda r: order.get(r["id"], 10**9))   # ghép cặp phải cùng thứ tự
    if not quiet:
        print(f"[ĐỌC] '{stage}': {len(rows)} mẫu")
    return rows


def stage_status():
    """Bảng trạng thái: nấc nào đã chạy, kết quả bao nhiêu."""
    print(f"\n{'─'*76}")
    print(f"  TIẾN ĐỘ — {RESULT_DIR}")
    print(f"{'─'*76}")
    print(f"  {'nấc':<22}{'':<4}{'n':>5}{'EA':>9}{'PA_strict':>11}{'chạy lúc':>16}")
    done = 0
    for stage, label in LADDER:
        mp = stage_path(stage, "meta")
        if not os.path.exists(mp):
            print(f"  {stage:<22}{'⊘':<4}{'—':>5}{'—':>9}{'—':>11}{'chưa chạy':>16}")
            continue
        m = json.load(open(mp, encoding="utf-8"))
        mt = m.get("metrics", {})
        done += 1
        print(f"  {stage:<22}{'✓':<4}{m.get('n','?'):>5}{mt.get('EA',0):>9.4f}"
              f"{mt.get('PA_strict',0):>11.4f}{m.get('stamp','?'):>16}")
    print(f"{'─'*76}\n  {done}/{len(LADDER)} nấc đã có kết quả")
    return done


_env = "Colab" if ON_COLAB else "máy cá nhân"
print(f"[MÔI TRƯỜNG] {_env} | vinumqa v{__import__('vinumqa').__version__}")
print(f"[REPO]  {REPO_DIR}")
print(f"[RA]    {OUTPUT_DIR}")
print(f"          ├─ stages/     kết quả từng nấc (jsonl + meta)")
print(f"          ├─ logs/       output thô của model")
print(f"          └─ artifacts/  playbook, adapter, biểu đồ")
print(f"[DỮ LIỆU] {DATA_DIR}")
print(f"          train={len(train_all)} valid={len(valid_all)} test={len(test_all)}")
stage_status()

In [ ]:
# ═══ Self-test: chạy TRƯỚC khi tốn GPU ═══
# Cell này đỏ thì dừng lại — mọi con số PA/EA sau đó sẽ vô nghĩa.

# (1) Ô cài đặt có thật sự cài được không. Đọc metadata nên nhanh, không phải import.
#     Kiểm ở đây để lỗi pip lộ ra trong 1 giây, thay vì 20 phút nữa lúc nạp model.
from importlib.metadata import version as _ver, PackageNotFoundError as _NoPkg
_goi = {}
for _p in ("vllm", "unsloth", "transformers", "trl", "peft", "torch"):
    try:
        _goi[_p] = _ver(_p)
    except _NoPkg:
        _goi[_p] = None
print("[GÓI] " + " | ".join(f"{k}={v}" for k, v in _goi.items() if v))
# vLLM phải khớp CUDA của torch, nếu không unsloth CHẶN import dù gói vẫn có mặt —
# lúc đó cell nạp model báo "No module named 'vllm'" một cách khó hiểu.
# Wheel khớp CUDA có đuôi "+cuXXX" trong số phiên bản; bản PyPI thì không.
if _goi.get("vllm") and "+cu" not in _goi["vllm"]:
    print("[GÓI] ⚠ vllm=" + _goi["vllm"] + " là bản PyPI (dựng cho CUDA 13). Nếu cell nạp "
          "model báo \"No module named 'vllm'\" thì cài lại bằng wheel khớp CUDA — "
          "dòng WARNING của unsloth in sẵn URL đúng.")

_thieu = [k for k, v in _goi.items() if v is None]
if _thieu:
    raise RuntimeError(
        "Thiếu gói: " + ", ".join(_thieu) + " — ô cài đặt (cell #1) đã thất bại.\n\n"
        "Cách chữa: mở Cửa sổ dòng lệnh (góc dưới trái), chạy\n"
        "    pip install -U unsloth vllm\n"
        "xem lỗi thật, xong Restart session rồi chạy lại TỪ CELL #2 (bỏ qua cell #1).")

# vLLM 0.23 cấm toàn bộ transformers 5.x. Gói nào đó nâng lên 5 thì chặn ngay tại đây,
# đừng để phát hiện sau 4 phút nạp model. (sentence-transformers ≥ 6 là thủ phạm hay gặp.)
if str(_goi["transformers"]).split(".")[0] != "4":
    raise RuntimeError(
        "transformers=" + str(_goi["transformers"]) + " — vLLM 0.23 chỉ chạy với "
        "transformers 4.x, gói nào đó đã nâng nó lên.\n"
        "Chữa: pip install \"transformers==4.57.6\" rồi Restart session.")

# (2) Executor có tái tạo đúng nhãn vàng không.
_ok = sum(dsl.check_ea(dsl.execute_program(s["qa"]["program"], s.get("table") or []),
                       s["qa"].get("exe_ans")) for s in test_all)
print(f"[SELF-TEST] executor tái tạo exe_ans trên test: {_ok}/{len(test_all)}")
assert _ok / len(test_all) > 0.99, "Executor không tái tạo được nhãn vàng — DỪNG."
assert dsl.execute_program("divide(5310, add(1, 0.15))", []) is None   # lồng nhau
assert dsl.check_pa("add(1, 2)", "add(2, 1)")[0]                       # giao hoán
assert dsl.check_ea(0.6066481994, "0.60665")                           # làm tròn 5 chữ số
print("[SELF-TEST] ✅ executor / PA / EA đạt")

## §2. Model

In [ ]:
# ═══════════════ MODEL — Qwen3-8B 4-bit ═══════════════
# Tham số sinh, cố định ở mọi nấc để hiệu số quy được cho kỹ thuật:
#   load_in_4bit=True, fast_inference=True, temperature=0.1
# max_tokens thì KHÔNG giữ: nâng 3000 → 8192 vì ở mức cũ 5–10 % mẫu bị cắt giữa lúc
# suy nghĩ, mất trắng. Xem lý do đầy đủ ở ô cấu hình GPU.
MODEL_NAME = "unsloth/Qwen3-8B"
MODEL_TAG  = "Qwen3-8B"

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Không thấy GPU. Runtime → Change runtime type → L4 GPU.")
_GPU, _VRAM = torch.cuda.get_device_name(0), torch.cuda.get_device_properties(0).total_memory/1024**3
_CC = torch.cuda.get_device_capability(0)
if _CC[0] < 7:
    raise RuntimeError(f"{_GPU} (CC {_CC[0]}.{_CC[1]}) không chạy được vLLM. "
                       f"Runtime → Change runtime type → A100 GPU.")

# ═══ Tham số ẢNH HƯỞNG KẾT QUẢ — CỐ ĐỊNH trên mọi GPU ═══
# Trước đây max_seq đổi theo GPU (A100 15000 / L4 13500) nên hai máy cho kết quả
# không so thẳng được. Giờ khoá cứng: đổi GPU chỉ đổi tốc độ, không đổi đầu vào.
#
# TRẦN SINH = 4096. Đây là mức ĐO ĐƯỢC là tối ưu, không phải chọn bừa:
#     nấc 2, cùng prompt, cùng GPU, chỉ khác trần —
#       4096 → 30 lượt bị cắt | 28 mẫu mất | EA 0.6479 | 300 mẫu đúng
#       8192 → 30 lượt        | 28 mẫu     | EA 0.6479 | 300 mẫu đúng
#     Gấp đôi ngân sách cứu ĐÚNG 0 mẫu. Số mẫu vượt trần không phụ thuộc trần, nên 4096
#     đã qua điểm bão hoà; 8192 chỉ tốn thêm thời gian. (Dưới 4096 thì mất thêm mẫu.)
#
# ~6 % mẫu vẫn chạm trần — nay KHÔNG bỏ mặc nữa: run_pipeline vớt chúng bằng một lượt
# sinh lại với suy nghĩ TẮT (xem `vot_mau_bi_cat`). Đó mới là cách chữa, không phải trần.
#
# max_seq 17000 theo ngân sách (neo vào phép đo thật bằng tokenizer):
#     prompt bước 2 xấu nhất = 7464 + 4096 = 11560
#     ngân sách              = 17000 − 4096 = 12904   → dư 1344 token
# Ô §3 đo lại bằng tokenizer thật và tự cắt ngữ cảnh + báo động nếu tính sai.
#
# ⚠ ĐỪNG nâng tiếp. Đã có phép so SẠCH: nấc 2 chạy hai lần với CÙNG prompt engineered,
# cùng model, cùng GPU, chỉ khác trần token —
#     trần 4096 → 30 lượt sinh bị cắt | 28 mẫu mất trắng | EA 0.6479 | 300 mẫu đúng
#     trần 8192 → 30 lượt             | 28 mẫu           | EA 0.6479 | 300 mẫu đúng
# Gấp đôi ngân sách cứu được ĐÚNG 0 mẫu, đổi lại ~50 % thời gian (10,9 → 16,4 phút).
#
# Số mẫu vượt ngân sách KHÔNG phụ thuộc ngân sách → những lượt đó thực tế không có điểm
# dừng. Mà chúng cũng không lặp (§4 đo trung vị lặp = 0.0 ở nấc 2), nên repetition_penalty
# cũng không phải thuốc. Coi đây là sàn ~6 %, đều ở mọi nấc: ghi nhận rồi bỏ qua.
TEMPERATURE, MAX_TOKENS = 0.1, 4096
MAX_SEQ_LENGTH = 17000
REPETITION_PENALTY = 1.0

# ═══ Tham số chỉ ảnh hưởng TỐC ĐỘ — chỉnh theo VRAM ═══
if _VRAM < 20:
    raise RuntimeError(
        f"{_GPU} chỉ {_VRAM:.0f} GB — không đủ cho max_seq={MAX_SEQ_LENGTH}.\n"
        f"Hạ max_seq xuống thì kết quả KHÔNG so được với các nấc khác, nên thà dừng "
        f"còn hơn ra một con số không dùng được. Đổi sang L4 hoặc A100.")
# util giữ 0.85 (hạ từ 0.88 sau một lần vLLM không dựng nổi engine vì VRAM còn sót).
# MAX_NUM_SEQS trả về mức cũ được vì max_seq đã từ 25000 xuống 17000, áp lực KV giảm hẳn.
#
# BATCH_SIZE = 512 để 497 mẫu vào ĐÚNG MỘT LÔ. Đo từ log thật: lô 400 mẫu chạy
# 1,88 s/mẫu, lô 97 mẫu còn lại chạy 2,83 s/mẫu — chậm hơn 50 % vì không lấp đầy GPU mà
# vẫn phải đợi mẫu dài nhất. Gộp một lô tiết kiệm ~1,5 phút MỖI lượt sinh; nấc 4 và nấc 5
# có nhiều lượt nên cộng lại đáng kể.
elif _VRAM < 30:                           # L4 24GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.86, 16, 512
elif _VRAM < 60:                           # A100 40GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 48, 512
else:                                      # A100 80GB
    GPU_MEM_UTIL, MAX_NUM_SEQS, BATCH_SIZE = 0.85, 128, 512
DTYPE = torch.float16 if _CC[0] < 8 else None

print(f"[GPU] {_GPU} | {_VRAM:.1f} GB | CC {_CC[0]}.{_CC[1]}")
print(f"[CFG] max_seq={MAX_SEQ_LENGTH} max_tokens={MAX_TOKENS} temp={TEMPERATURE} "
      f"(cố định mọi GPU) | batch={BATCH_SIZE} max_num_seqs={MAX_NUM_SEQS} (theo VRAM)")

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import unsloth
from unsloth import FastLanguageModel
from vllm import SamplingParams

torch.manual_seed(RANDOM_SEED); torch.cuda.manual_seed_all(RANDOM_SEED)

import shutil as _sh

# Đặt True nếu model tải về bị thiếu trọng số: tắt hf_transfer thì tải chậm hơn vài phút
# nhưng có kiểm tra và tải tiếp được. Lưu ý: `export` trong Cửa sổ dòng lệnh KHÔNG tới
# được kernel notebook — phải đặt ở đây.
TAI_CHAM_CHO_CHAC = False
if TAI_CHAM_CHO_CHAC:
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    print("[MODEL] đã tắt hf_transfer — tải chậm hơn nhưng chắc hơn")

_free = _sh.disk_usage("/").free / 1024**3
_t_nap = time.time()
print(f"[MODEL] Đang tải {MODEL_NAME} ... (đĩa trống {_free:.0f} GB)")
if _free < 15:
    print("[MODEL] ⚠ dưới 15 GB trống — model ~15 GB, tải dễ đứt giữa chừng.")
print("[MODEL] ⏳ Mất 4–7 PHÚT. Tải xong rồi vLLM còn dựng CUDA graph — đoạn đó")
print("[MODEL]    KHÔNG có thanh tiến trình, nhìn như treo nhưng không phải.")
print("[MODEL]    Muốn biết còn sống: xem MỐC GIỜ ở các dòng INFO bên dưới. Nó nhích")
print("[MODEL]    lên là đang chạy. Đứng im quá 10 phút mới đáng nghi.")

# enable_prefix_caching: system prompt (~1 800 token) GIỐNG HỆT ở cả 497 request, nên
# vLLM chỉ cần prefill nó một lần rồi dùng lại. Tiết kiệm phần lớn thời gian prefill.
# Không đổi token sinh ra — mỗi request vẫn có seed riêng.
_NAP_KW = dict(model_name     = MODEL_NAME,
               dtype          = DTYPE,
               max_seq_length = MAX_SEQ_LENGTH,
               load_in_4bit   = True,
               fast_inference = True)
try:                                  # bản unsloth cũ không nhận tham số này
    import inspect as _insp
    if "enable_prefix_caching" in _insp.signature(
            FastLanguageModel.from_pretrained).parameters:
        _NAP_KW["enable_prefix_caching"] = True
except Exception:                                    # noqa: BLE001
    pass
NAP_AN_TOAN = False          # True = đã phải lùi về chế độ an toàn, có ghi vào meta

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        **_NAP_KW, gpu_memory_utilization=GPU_MEM_UTIL, max_num_seqs=MAX_NUM_SEQS)
except (RuntimeError, ValueError) as _e:
    # ── vLLM dựng engine hỏng vì CUDA ──
    # KHÔNG phải tải model hỏng: model đã nằm trên đĩa rồi. Lỗi ở bước cấp phát KV cache
    # và dựng CUDA graph — thường do VRAM trống ít hơn lần trước (GPU khác, hoặc tiến
    # trình cũ còn giữ bộ nhớ), khiến số block KV tính ra quá nhỏ.
    if "CUDA error" in str(_e) or "invalid argument" in str(_e):
        print("[MODEL] ⚠ vLLM KHÔNG dựng được engine (CUDA error).")
        print(f"[MODEL]   Đang dùng: max_seq={MAX_SEQ_LENGTH} util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}")
        try:
            _free, _tot = torch.cuda.mem_get_info()
            print(f"[MODEL]   VRAM trống: {_free/1024**3:.1f}/{_tot/1024**3:.1f} GB"
                  + ("   ← ĐÃ BỊ CHIẾM. Restart session rồi chạy lại TỪ Ô #2."
                     if _free / _tot < 0.9 else ""))
        except Exception:                                    # noqa: BLE001
            pass
        print("[MODEL]   Thử lại ở CHẾ ĐỘ AN TOÀN: bỏ CUDA graph, hạ VRAM và số chuỗi.")
        print("[MODEL]   Ba thứ đó chỉ đổi TỐC ĐỘ — mỗi request đã có seed riêng nên")
        print("[MODEL]   thành phần lô không ảnh hưởng token sinh ra.")
        gc.collect()
        torch.cuda.empty_cache()
        _an = dict(gpu_memory_utilization=min(GPU_MEM_UTIL, 0.80),
                   max_num_seqs=max(8, MAX_NUM_SEQS // 4))
        try:
            model, tokenizer = FastLanguageModel.from_pretrained(
                **_NAP_KW, enforce_eager=True, **_an)
        except TypeError:                 # bản unsloth không nhận enforce_eager
            model, tokenizer = FastLanguageModel.from_pretrained(**_NAP_KW, **_an)
        GPU_MEM_UTIL = _an["gpu_memory_utilization"]
        MAX_NUM_SEQS = _an["max_num_seqs"]
        NAP_AN_TOAN = True
        print(f"[MODEL] ✅ nạp được ở chế độ an toàn (util={GPU_MEM_UTIL} "
              f"max_num_seqs={MAX_NUM_SEQS}) — chậm hơn, kết quả không đổi.")
    # ── Thiếu trọng số: shard safetensors tải dở còn trong cache ──
    elif "not initialized from checkpoint" in str(_e):
        raise RuntimeError(
            "Model thiếu trọng số — bản tải dở trong cache HuggingFace.\n\n"
            "Bước 1 — xoá cache. Mở Cửa sổ dòng lệnh (góc dưới trái):\n"
            "    rm -rf ~/.cache/huggingface/hub/models--unsloth--Qwen3-8B*\n"
            "    df -h / | tail -1          # kiểm luôn, cần ≥ 20 GB trống\n\n"
            "Bước 2 — đặt TAI_CHAM_CHO_CHAC = True ở ĐẦU CHÍNH Ô NÀY.\n"
            "    (`export` trong terminal không tới được kernel notebook.)\n\n"
            "Bước 3 — Restart session, chạy lại TỪ CELL #2 (bỏ qua ô cài đặt).\n\n"
            "Hỏng y hệt lần nữa thì không phải do tải: khi đó là bản 4-bit của unsloth "
            "không khớp bộ nạp của vLLM, phải đổi phiên bản chứ không phải tải lại.") from _e
    else:
        raise

SAMPLING = SamplingParams(temperature=TEMPERATURE, max_tokens=MAX_TOKENS,
                          repetition_penalty=REPETITION_PENALTY, seed=RANDOM_SEED)
LORA_REQUEST = None          # nấc 3 trở đi có thể gán adapter đã SFT vào đây

from collections import Counter as _Counter
LY_DO_DUNG = _Counter()          # finish_reason: "stop" = tự kết thúc, "length" = BỊ CẮT
LY_DO_THEO_BUOC = {}             # desc → Counter riêng, để tách bước 1 với bước 2


def ty_le_bi_cat():
    """Phần trăm lượt sinh bị cắt vì chạm max_tokens, TÍNH TỪ LẦN reset gần nhất."""
    t = sum(LY_DO_DUNG.values())
    return (LY_DO_DUNG.get("length", 0) / t) if t else 0.0


def bi_cat_theo_buoc():
    """Tỉ lệ bị cắt TÁCH RIÊNG cho bước 1 và bước 2.

    Phải tách vì prompt bước 2 (self-eval ở nấc 4, ACE ở nấc 5) chứa NGUYÊN lời giải
    bước 1, nên dài hơn bước 1 rất nhiều. Bước 2 bị cắt nhiều hơn nghĩa là phương pháp
    bị PHA LOÃNG — mất cơ hội sửa, chứ không phải sửa sai. Con số gộp chung không phân
    biệt được hai chuyện đó.

    Gom theo đuôi của desc ("vòng3/step1" và "vòng7/step1" cùng vào "step1").
    """
    gom = {}
    for k, c in LY_DO_THEO_BUOC.items():
        gom.setdefault(k.rsplit("/", 1)[-1], _Counter()).update(c)
    return {b: {"n": sum(c.values()), "bi_cat": c.get("length", 0),
                "ty_le": round(c.get("length", 0) / max(1, sum(c.values())), 4)}
            for b, c in sorted(gom.items())}


def in_bi_cat_theo_buoc():
    d = bi_cat_theo_buoc()
    if not d:
        return
    # In cả khi chỉ có MỘT bước: nấc 1 và 2 cũng cần biết tỉ lệ chạm trần của mình,
    # nếu không thì mãi tới nấc 4 mới thấy con số đó.
    print("   Bị cắt vì trần token, tách theo bước:")
    for b, v in d.items():
        print(f"     {b:<10}{v['ty_le']:>7.1%}  ({v['bi_cat']}/{v['n']} lượt)")
    if "step2" in d and "step1" in d and d["step2"]["ty_le"] > d["step1"]["ty_le"] + 0.02:
        print("     ⚠ bước 2 bị cắt nhiều hơn bước 1 → hiệu quả của phương pháp đang bị")
        print("       PHA LOÃNG (mất cơ hội sửa). Hiệu số đo được là cận DƯỚI.")


def dat_lai_bo_dem():
    """Gọi ngay trước mỗi nấc. Không gọi thì tỉ lệ là cộng dồn cả phiên — gồm cả
    lượt warmup và (ở nấc 5) toàn bộ pha A, không phản ánh nấc đang đo."""
    LY_DO_DUNG.clear()
    LY_DO_THEO_BUOC.clear()


def generate(prompts, sampling_params=None, desc=None, batch_size=None):
    """Sinh theo lô qua vLLM — như vòng lặp trong notebook cũ."""
    if not prompts:
        return []
    sp = sampling_params or SAMPLING
    bs = batch_size or BATCH_SIZE
    outs, t0 = [], time.time()
    nb = (len(prompts) + bs - 1) // bs
    for i in range(nb):
        chunk = prompts[i*bs:(i+1)*bs]
        kw = {"sampling_params": sp}
        if LORA_REQUEST is not None:
            kw["lora_request"] = LORA_REQUEST
        _res = model.fast_generate(chunk, **kw)
        for o in _res:                      # đếm lý do dừng để biết có bị cắt không
            for _x in o.outputs:            # sinh nhiều mẫu thì đếm CẢ K mẫu
                _r = getattr(_x, "finish_reason", "?")
                LY_DO_DUNG[_r] += 1
                if desc:
                    LY_DO_THEO_BUOC.setdefault(desc, _Counter())[_r] += 1
        # 1 mẫu → trả chuỗi (y như cũ); nhiều mẫu → trả list[str] cho self-consistency.
        outs.extend((o.outputs[0].text if len(o.outputs) == 1
                     else [_x.text for _x in o.outputs]) for o in _res)
        if desc:
            el = time.time() - t0
            print(f"    {desc}: lô {i+1}/{nb} | {el:.0f}s | "
                  f"ETA {el/(i+1)*(nb-i-1):.0f}s", end="\r")
    gc.collect(); torch.cuda.empty_cache()
    if desc:
        _b = LY_DO_THEO_BUOC.get(desc, _Counter())
        _c, _n = _b.get("length", 0), max(1, sum(_b.values()))
        print(f"    {desc}: xong {len(prompts)} prompt trong {time.time()-t0:.0f}s"
              f" | bị cắt vì trần token: {_c/_n:.1%} ({_c} lượt)" + " "*8)
    return outs

_t = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f"[MODEL] ✅ sẵn sàng sau {(time.time()-_t_nap)/60:.1f} phút | "
      f"VRAM {_t - torch.cuda.mem_get_info()[0]/1024**3:.1f}/{_t:.1f} GB")
_ = generate(["xin chào"], SamplingParams(temperature=0, max_tokens=4))
print("[WARMUP] ✅")

In [ ]:
prompt_kit = PromptKit(tokenizer=tokenizer, model_name=MODEL_NAME)
PROMPT_LEVEL = "engineered"
USE_SELFEVAL = False

_think = getattr(prompt_kit, "enable_thinking", None)
print(f"[PROMPT] mức = {PROMPT_LEVEL} | self-eval = {USE_SELFEVAL} | "
      f"suy nghĩ = {'template tự quyết (Qwen3: BẬT)' if _think is None else _think}")
if _think is False:
    print("[PROMPT] ⚠ suy nghĩ đang TẮT — lệch bản tham chiếu, PA sẽ hụt "
          "~10 điểm. Dấu hiệu: 497 mẫu chạy xong trong ~1 phút.")
print(f"         thang lồng nhau: basic={len(prompt_kit.BASIC_SYSTEM_PROMPT)} ký tự"
      f" ⊂ no_fewshot={len(prompt_kit.NO_FEWSHOT_SYSTEM_PROMPT)}"
      f" ⊂ engineered={len(prompt_kit.ENGINEERED_SYSTEM_PROMPT)}"
      f" | self-eval={len(prompt_kit.SELF_EVAL_SYSTEM_PROMPT)}")

# Đo bằng tokenizer THẬT trên 40 mẫu có ngữ cảnh DÀI NHẤT
BUDGET = MAX_SEQ_LENGTH - MAX_TOKENS
_clen = lambda s: (len(" ".join(s.get("pre_text") or [])) +
                   len(" ".join(s.get("post_text") or [])) + len(str(s.get("table") or "")))
_probe = sorted(test_all, key=_clen, reverse=True)[:40]
_bul = "\n".join(["- Khi hỏi tốc độ tăng trưởng, dùng subtract(gia_tri_moi, gia_tri_cu), "
                  "divide(#0, gia_tri_cu)."] * 7)
# Lời giải bước 1 dài nhất có thể là đúng MAX_TOKENS token (model sinh chạm trần).
# Phải đo ở mức đó, không thì bật suy nghĩ vào là prompt bước 2 tràn ngân sách.
_unit = "Phân tích chi tiết từng bước của bảng số liệu. "
_prev = _unit * max(1, MAX_TOKENS // max(1, len(tokenizer(_unit).input_ids)))
_prev += "\n```plaintext\nprogram: divide(1,2)\nanswer: 0.5\n```"

def _measure():
    a = [len(tokenizer(prompt_kit.step1(s, _bul, level=PROMPT_LEVEL)).input_ids)
         for s in _probe]
    b = ([len(tokenizer(prompt_kit.step2(s, _prev, _bul)).input_ids) for s in _probe]
         if USE_SELFEVAL else [0])
    return a, b

_a, _b = _measure()
print(f"[PROMPT] (40 mẫu dài nhất) step1 max={max(_a)} | step2 max={max(_b)} | "
      f"ngân sách={BUDGET}")

if max(max(_a), max(_b)) > BUDGET:
    prompt_kit.max_prev_chars = 3000
    _cap = _clen(_probe[0])
    for _ in range(6):
        _cap = int(_cap * 0.80)
        prompt_kit.max_ctx_chars = max(1200, _cap)
        _a, _b = _measure()
        if max(max(_a), max(_b)) <= BUDGET:
            break
    assert max(max(_a), max(_b)) <= BUDGET, "Không cắt đủ — giảm MAX_TOKENS hoặc dùng GPU lớn hơn."
    _hit = sum(1 for s in test_all if _clen(s) > prompt_kit.max_ctx_chars)
    print(f"[PROMPT] ⚠ đã bật cắt ngữ cảnh (max_ctx_chars={prompt_kit.max_ctx_chars}); "
          f"{_hit}/{len(test_all)} mẫu bị cắt ({_hit/len(test_all)*100:.1f}%)")
    print(f"[PROMPT]   GHI LẠI con số này khi báo cáo.")
else:
    print("[PROMPT] ✅ mọi prompt đều lọt ngân sách, không cần cắt")

## §3. Định nghĩa ma trận

Mỗi ô ánh xạ sang tên nấc đã dùng ở các notebook trước, nên ô nào đã có kết quả thì đọc lại
được ngay. Hai ô `E+A` và `F+A` là mới — chúng tách phần đóng góp của ACE **khi không có**
self-eval, tức là cho biết ACE có thể **thay thế** self-eval hay không (rẻ hơn một nửa vì
chỉ một lượt sinh).

In [ ]:
# (mã ô, dùng SFT, dùng self-eval, dùng ACE, tên nấc tương ứng)
MATRIX = [
    ("E",     False, False, False, "02_prompt_eng"),
    ("E+A",   False, False, True,  "06_comb_E_A"),
    ("E+S",   False, True,  False, "04_selfeval_base"),
    ("E+S+A", False, True,  True,  "05_ace_base"),
]
CELL_LABEL = {c: c for c, *_ in MATRIX}
NICE = {"E": "prompt", "E+A": "prompt+ACE", "E+S": "prompt+selfeval",
        "E+S+A": "prompt+selfeval+ACE"}

# Cột "dùng SFT" vẫn nằm trong tuple (luôn False) để mọi vòng lặp bên dưới khỏi phải
# đổi hình dạng — nhưng KHÔNG ô nào bật nó nữa. Nấc 3 đo được EA 0,6761 so với 0,6781
# của nấc 2: McNemar p = 1,000, KTC [-0,036; +0,034], 41 mẫu đúng thêm đổi 42 mẫu hỏng
# đi. Giữ 4 ô SFT là tiêu thêm ~4 giờ GPU để đo lại một số 0, và làm loãng tác động
# chính của hai yếu tố còn lại.

print(f"\n{'ô':<8}{'self-eval':>11}{'ACE':>6}  {'nấc':<20}{'trạng thái'}")
print("-" * 68)
TODO = []
for code, use_sft, use_se, use_ace, stage in MATRIX:
    have = os.path.exists(stage_path(stage, "jsonl"))
    if not have:
        TODO.append((code, use_sft, use_se, use_ace, stage))
    print(f"{code:<8}{'✓' if use_se else '·':>11}"
          f"{'✓' if use_ace else '·':>6}  {stage:<20}"
          f"{'đã có — đọc lại' if have else 'CẦN CHẠY'}")

_units = sum((2 if se else 1) for _c, _f, se, _a, _s in TODO)
print(f"\n  {len(MATRIX)-len(TODO)}/{len(MATRIX)} ô đã có sẵn.")
print(f"  Cần chạy {len(TODO)} ô ≈ {_units} lượt × {len(test_all)} mẫu "
      f"≈ {_units*20}–{_units*30} phút trên L4.")

## §4. Nạp playbook ACE

Playbook do `05_ace.ipynb` học ra. Hai ô mới (`E+A`, `F+A`) dùng **cùng playbook đó** nhưng
bỏ bước self-eval.

> Lưu ý phải nêu khi báo cáo: playbook được học **trong điều kiện có self-eval**. Đem dùng ở
> cấu hình không self-eval là hơi lệch phân phối. Đây là đánh đổi có chủ ý để khỏi phải chạy
> thêm một pha A (~2 giờ); nếu cần số liệu thật chặt thì học lại playbook với
> `use_selfeval=False` ở `05_ace.ipynb`.

In [ ]:
def find_playbook(prefer_sft):
    """Ưu tiên playbook khớp model nền, không có thì lấy cái còn lại."""
    order = (["playbook_ace_sft.txt", "playbook_ace_base.txt"] if prefer_sft
             else ["playbook_ace_base.txt", "playbook_ace_sft.txt"])
    for name in order:
        for d in (OUTPUT_DIR, ARTIFACT_DIR):
            p = os.path.join(d, name)
            if os.path.exists(p):
                return p, name
    return None, None

_p, _n = find_playbook(False)
if _p is None:
    print("[PLAYBOOK] ⚠ chưa thấy playbook nào — bỏ mọi ô có ACE.")
    print("           Chạy 05_ace.ipynb trước.")
    MATRIX = [m for m in MATRIX if not m[3]]
    TODO = [t for t in TODO if not t[3]]
    PLAYBOOK_BASE = ""
else:
    PLAYBOOK_BASE = open(_p, encoding="utf-8").read()
    print(f"[PLAYBOOK] model gốc → {_n} ({len(pb_mod.all_bullets(PLAYBOOK_BASE))} bullet)")

embedder = pb_mod.Embedder()
retriever = pb_mod.Retriever(embedder=embedder, k_tier1=3, k_tier2=4)

# Khôi phục Tier-1 nếu 05_ace có lưu tiến độ
for _tag in ("ace_base",):
    _pr = os.path.join(OUTPUT_DIR, f"progress_{_tag}.json")
    if os.path.exists(_pr):
        _st = json.load(open(_pr, encoding="utf-8"))
        retriever.tier1_ids.update(_st.get("tier1_ids", []))
print(f"[ACE] embedding={embedder.name} | Tier-1: {len(retriever.tier1_ids)} bullet")

## §5. Chạy các ô còn thiếu

In [ ]:
RESULTS = {}

# đọc lại những ô đã có
for code, use_sft, use_se, use_ace, stage in MATRIX:
    rows = load_stage(stage, quiet=True)
    if rows is not None:
        RESULTS[code] = (rows, pipeline.summarize(rows, NICE[code]), stage)
        print(f"  ✓ {code:<8} đọc lại từ '{stage}'  EA={RESULTS[code][1]['EA']:.4f}")

# chạy những ô còn thiếu
for code, use_sft, use_se, use_ace, stage in TODO:
    LORA_REQUEST = None
    pb = PLAYBOOK_BASE if use_ace else ""
    print(f"\n{'═'*74}\n  Ô {code} — {NICE[code]} | self-eval={use_se} "
          f"ACE={use_ace}\n{'═'*74}")
    dat_lai_bo_dem()          # tỉ lệ bị cắt tính riêng cho từng ô ma trận
    _t0 = time.time()
    rows = pipeline.run_pipeline(test_all, prompt_kit, generate,
                                 prompt_level="engineered", use_selfeval=use_se,
                                 playbook=pb, retriever=retriever if use_ace else None,
                                 sp_step1=SAMPLING, sp_step2=SAMPLING, desc=code)
    m = pipeline.summarize(rows, NICE[code])
    m["minutes"] = round((time.time()-_t0)/60, 1)
    pipeline.print_summary(m)
    in_bi_cat_theo_buoc()
    RESULTS[code] = (rows, m, stage)
    save_stage(stage, rows, m, extra={"cell": code, "sft": use_sft,
                                      "self_eval": use_se, "ace": use_ace,
                                      "prompt_level": "engineered"})

LORA_REQUEST = None
print(f"\n✅ Ma trận có {len(RESULTS)}/{len(MATRIX)} ô.")

## §5b. Hai ô MỤC TIÊU với cấu hình tốt nhất (tuỳ chọn)

Ma trận 2×2 ở trên cố tình chạy **đúng cấu hình của thang bậc** (1 mẫu, `temperature`
0.1, ví dụ cố định) — chỉ như vậy thì tác động chính và tương tác của self-eval × ACE
mới sạch.

Nhưng **mục tiêu 70/70 không bắt buộc phải dùng cấu hình đó**. Nếu bước 4 cho thấy
self-consistency hoặc ví dụ động vượt sàn nhiễu, ô mục tiêu nên được chạy ở cấu
hình tốt nhất. Đó là việc của khối dưới đây.

| | ma trận 2×2×2 | hai ô mục tiêu |
|---|---|---|
| để làm gì | đo đóng góp self-eval × ACE | **đạt 70/70** |
| cấu hình sinh | 1 mẫu, temp 0.1, ví dụ cố định | K mẫu, temp cao, ví dụ truy hồi, sửa-khi-lỗi |
| tên nấc | `06_comb_E_A` | `04_selfeval_base_moi` |

Hai nhóm ghi ra **tên nấc khác nhau**, nên không bao giờ đè lên nhau và bảng kiểm công
bằng ở `07` không báo động oan.

⚠ Để `CHAY_O_TOT_NHAT = False` nếu bước 4 cho Δ nằm trong sàn nhiễu — khi đó chạy thêm
chỉ tốn ~160 phút GPU mà không mua được gì.

In [ ]:
# ═══════════ HAI Ô MỤC TIÊU, CẤU HÌNH TỐT NHẤT ═══════════
# Mặc định BẬT: chạy một lượt là có đủ thông tin, không phải quay lại notebook lần nữa.
# Đặt False nếu muốn tiết kiệm ~160 phút và đã biết chắc phương pháp mới nằm trong sàn
# nhiễu (xem "SELF-CONSISTENCY THEO k" và "VÍ DỤ ĐỘNG ĐÓNG GÓP" ở 07 §5d).
CHAY_O_TOT_NHAT = True

# Khớp với notebook 08. Đổi ở đây thì không so thẳng với nấc 08/09 được nữa.
SO_MAU_TOT_NHAT, TEMP_TOT_NHAT, SO_VI_DU_TOT_NHAT = 5, 0.7, 3

if not CHAY_O_TOT_NHAT:
    print("[Ô TỐT NHẤT] tắt — ma trận 2×2×2 ở trên là kết quả cuối.")
    print("             Bật lên nếu bước 4 cho thấy phương pháp mới ăn tiền.")
else:
    from vinumqa.fewshot import KhoViDu
    _KHO = KhoViDu(train_all)
    _SP = SamplingParams(temperature=TEMP_TOT_NHAT, top_p=0.95, max_tokens=MAX_TOKENS,
                         n=SO_MAU_TOT_NHAT, repetition_penalty=REPETITION_PENALTY,
                         seed=RANDOM_SEED)
    print(f"[Ô TỐT NHẤT] K={SO_MAU_TOT_NHAT} temp={TEMP_TOT_NHAT} "
          f"| {SO_VI_DU_TOT_NHAT} ví dụ truy hồi | kho {len(_KHO)} mẫu")

    # Cổng kiểm sớm, y như notebook 08 — chặn 160 phút nếu n=K không có tác dụng.
    _thu = generate([prompt_kit.step1(s, level="engineered",
                                      vi_du_dong=_KHO.van_ban_vi_du(s["qa"]["question"],
                                                                    SO_VI_DU_TOT_NHAT))
                     for s in test_all[:2]], _SP, desc="cong-kiem")
    assert isinstance(_thu[0], list) and len(_thu[0]) == SO_MAU_TOT_NHAT, (
        f"xin {SO_MAU_TOT_NHAT} mẫu, nhận {len(_thu[0]) if isinstance(_thu[0], list) else 1}"
        f" — SamplingParams(n=) không có tác dụng, đừng chạy tiếp.")
    print(f"[Ô TỐT NHẤT] ✅ cổng kiểm: nhận đúng {len(_thu[0])} mẫu/prompt")

    # Ô mục tiêu = ô ma trận TỐT NHẤT chạy ở cấu hình tốt nhất. Chọn E+S chứ không phải
    # E+A, vì số đo đã có: ACE cho Δ = 0,00 so với self-eval (32 sửa đổi 32 hỏng) và
    # từng bullet đều net ≈ 0 — không có cơ chế nào khiến nó khác đi khi thêm 5 mẫu bỏ
    # phiếu. Còn self-eval cho +1,61 điểm (9 sửa / 1 hỏng, đối chứng trong-lượt), và
    # đây là TỔ HỢP DUY NHẤT chưa từng chạy: K=5 + ví dụ truy hồi + self-eval.
    _O_MUC_TIEU = [(c, f, s, a, st + "_moi") for c, f, s, a, st in MATRIX
                   if c in ("E+S",)]
    for code, use_sft, use_se, use_ace, stage in _O_MUC_TIEU:
        if load_stage(stage, quiet=True) is not None:
            print(f"  ✓ {code:<8} đã có '{stage}', bỏ qua")
            continue
        LORA_REQUEST = None
        pb = PLAYBOOK_BASE if use_ace else ""
        print(f"\n{'═'*74}\n  Ô {code} (TỐT NHẤT) — {NICE[code]} + K mẫu + ví dụ động"
              f"\n{'═'*74}")
        dat_lai_bo_dem()
        _t0 = time.time()
        _rows = pipeline.run_pipeline(
            test_all, prompt_kit, generate,
            prompt_level="engineered", use_selfeval=use_se,
            playbook=pb, retriever=retriever if use_ace else None,
            kho_vi_du=_KHO, n_vi_du=SO_VI_DU_TOT_NHAT,
            # Bước 2 là lượt SOÁT, không phải bài toán lấy mẫu: `run_pipeline`
            # chỉ dùng mẫu đầu của bước 2, nên để n=5 là vứt đi 4/5 lượt sinh.
            # `SAMPLING` (1 mẫu, temp 0.1) cũng đúng với cấu hình nấc 4 đã đo.
            sp_step1=_SP, sp_step2=SAMPLING, sua_khi_loi=True, desc=stage)
        _m = pipeline.summarize(_rows, NICE[code] + " (tốt nhất)")
        _m["minutes"] = round((time.time()-_t0)/60, 1)
        pipeline.print_summary(_m)
        in_bi_cat_theo_buoc()
        RESULTS[code + "*"] = (_rows, _m, stage)
        save_stage(stage, _rows, _m,
                   extra={"cell": code + "*", "sft": use_sft, "self_eval": use_se,
                          "ace": use_ace, "prompt_level": "engineered",
                          "so_mau": SO_MAU_TOT_NHAT, "temperature_moi": TEMP_TOT_NHAT,
                          "vi_du_dong": True, "so_vi_du": SO_VI_DU_TOT_NHAT,
                          "sua_khi_loi": True,
                          "tran_best_of_k": pipeline.tran_best_of_k(_rows)})
    LORA_REQUEST = None

    # Đường cong k cho hai ô mục tiêu — tính ngay, CPU.
    print(f"\n{'═'*84}\n  HAI Ô MỤC TIÊU THEO k\n{'═'*84}")
    print(f"{'ô':<10}" + "".join(f"{'k='+str(k):>11}" for k in range(1, SO_MAU_TOT_NHAT+1))
          + f"{'trần':>10}")
    for code in ("E+S*",):
        if code not in RESULTS:
            continue
        _r = RESULTS[code][0]
        print(f"{code:<10}" + "".join(
            f"{pipeline.summarize(pipeline.tu_nhat_quan(_r, test_all, k), 'x')['EA']:>11.4f}"
            for k in range(1, SO_MAU_TOT_NHAT+1))
            + f"{pipeline.tran_best_of_k(_r)['EA_tran']:>10.4f}")
    print("  (EA. So với ô cùng tên KHÔNG có dấu * để thấy phương pháp mới thêm bao nhiêu.)")

## §6. Ma trận kết quả

In [ ]:
_order = [c for c, *_ in MATRIX if c in RESULTS]


def ea(c):
    """EA của một ô. Cell 21 và 24 vẫn gọi hàm này — trước đây nó KHÔNG hề tồn tại,
    nên notebook nổ NameError ngay sau khi chạy xong toàn bộ phần GPU."""
    return RESULTS[c][1]["EA"]


def pa(c):
    return RESULTS[c][1]["PA_strict"]


print(f"\n{'═'*96}\n  MA TRẬN TỔ HỢP — Qwen3-8B, ViNumQA test ({len(test_all)} mẫu)\n{'═'*96}")
print(f"{'ô':<8}{'SE':>5}{'ACE':>5}  {'cấu hình':<24}{'EA':>9}{'PA_strict':>11}"
      f"{'lượt sinh':>11}{'phút':>8}")
print("-" * 96)
_spec = {c: (f, s, a) for c, f, s, a, _ in MATRIX}
for c in _order:
    f, s, a = _spec[c]
    m = RESULTS[c][1]
    print(f"{c:<8}{'✓' if s else '·':>5}{'✓' if a else '·':>5}  "
          f"{NICE[c]:<24}{m['EA']:>9.4f}{m['PA_strict']:>11.4f}"
          f"{(2 if s else 1):>11}{m.get('minutes', 0):>8.1f}")

_best = max(_order, key=lambda c: RESULTS[c][1]["EA"])
print(f"\n  Tốt nhất: ô {_best} ({NICE[_best]}) — EA {RESULTS[_best][1]['EA']:.4f}")

# ═══ CỔNG MỤC TIÊU: hai ô tổ hợp phải > 70 % CẢ EA LẪN PA ═══
MUC_TIEU = 0.70
# Gồm cả hai ô MỤC TIÊU chạy cấu hình tốt nhất (hậu tố *), nếu đã chạy.
_o_to_hop = [c for c in ("E+A", "E+S", "E+S*") if c in RESULTS]
if _o_to_hop:
    print(f"\n{'─'*96}\n  CỔNG MỤC TIÊU — EA > {MUC_TIEU:.0%} VÀ PA_strict > {MUC_TIEU:.0%}\n{'─'*96}")
    for c in _o_to_hop:
        m = RESULTS[c][1]
        _ok = m["EA"] > MUC_TIEU and m["PA_strict"] > MUC_TIEU
        # NICE chỉ có khoá của ô ma trận; ô mục tiêu có hậu tố * nên phải cắt đi.
        _ten = NICE[c.rstrip("*")] + (" + PP mới" if c.endswith("*") else "")
        print(f"  {c:<8}{_ten:<24}EA {m['EA']:.4f}  PA {m['PA_strict']:.4f}   "
              + ("✅ ĐẠT" if _ok else
                 f"⛔ còn thiếu EA {max(0, MUC_TIEU-m['EA'])*len(test_all):.0f} mẫu, "
                 f"PA {max(0, MUC_TIEU-m['PA_strict'])*len(test_all):.0f} mẫu"))
    _dat = [c for c in _o_to_hop
            if RESULTS[c][1]["EA"] > MUC_TIEU and RESULTS[c][1]["PA_strict"] > MUC_TIEU]
    if _dat:
        print(f"\n  ✅ ĐẠT MỤC TIÊU ở: {', '.join(_dat)}")
    if not _dat:
        print(f"\n  Chưa đạt thì xem bảng `vi_sao_sai` và `by_phep` ở notebook 07 để biết")
        print("  thiếu ở loại câu nào, đừng đoán.")


## §7. Tác động chính và tương tác

**Tác động chính** của một kỹ thuật = trung bình phần nó cộng thêm, lấy qua mọi thiết lập của
hai kỹ thuật kia. Con số này đáng tin hơn "Δ so với nấc trước" của chuỗi tuyến tính, vì nó
không phụ thuộc vào việc hai kỹ thuật kia đang bật hay tắt.

**Tương tác** = chênh lệch giữa hai lần đo tác động đó. Ví dụ với ACE:

```
ACE khi chưa SFT   = (E+A − E)     và   (E+S+A − E+S)
ACE khi đã SFT     = (F+A − F)     và   (F+S+A − F+S)
tương tác SFT×ACE  = trung bình nhóm sau − trung bình nhóm trước
```

* **Tương tác âm** → hai kỹ thuật **trùng nhau**: cái này đã học phần lớn những gì cái kia
  dạy được. Đây là kết quả đáng nghi ngờ nhất với cặp SFT × ACE, vì cả hai đều học từ train.
* **Tương tác ≈ 0** → **cộng dồn** được, dùng cả hai là hợp lý.
* **Tương tác dương** → **cộng hưởng**, hiếm nhưng đáng báo cáo.

In [ ]:
# ═══ Tác động chính và tương tác, CHO CẢ EA LẪN PA, kèm sai số ═══
# Bản trước chỉ tính EA và dùng ngưỡng cứng ±0.01 để phán "cộng hưởng"/"trùng nhau".
# 0.01 = 1 điểm EA. Đo thật cho thấy chỉ đổi card thôi đã chênh 2,21 điểm, nên ngưỡng
# cứng nào cũng là phán trên nhiễu.
# Nay mọi con số đều có KTC bootstrap lấy mẫu lại THEO MẪU (mọi ô chấm trên cùng 497
# mẫu nên đây là cách đúng), và kết luận chỉ đưa ra khi KTC không chứa 0.

def _co(c, key):
    """Vector cờ đúng/sai từng mẫu của một ô."""
    return [bool(r[key]) for r in RESULTS[c][0]]

def _cap_theo(factor):
    """Các cặp ô (TẮT → BẬT) chỉ khác đúng một yếu tố, và cả hai ô đều đã chạy."""
    out = []
    for c, f, s, a, _ in MATRIX:
        spec = {"sft": f, "se": s, "ace": a}
        if spec[factor] or c not in RESULTS:
            continue
        want = dict(spec); want[factor] = True
        for c2, f2, s2, a2, _ in MATRIX:
            if {"sft": f2, "se": s2, "ace": a2} == want and c2 in RESULTS:
                out.append((c, c2))
    return out

TEN = {"se": "Self-eval", "ace": "ACE"}
MAIN = {}
for key, nhan in (("ea", "EA"), ("pa_strict", "PA_strict")):
    print(f"\n{'═'*92}\n  TÁC ĐỘNG CHÍNH — {nhan} (trung bình qua mọi thiết lập còn lại)\n{'═'*92}")
    print(f"{'kỹ thuật':<14}{'số cặp':>8}{'Δ':>10}{'KTC 95%':>22}   kết luận")
    for fac in ("se", "ace"):
        cap = _cap_theo(fac)
        if not cap:
            print(f"{TEN[fac]:<14}{'—':>8}{'(thiếu ô)':>10}")
            continue
        r = stats.ktc_hieu_ung([(_co(a, key), _co(b, key)) for a, b in cap])
        d, lo, hi = r
        MAIN[f"{fac}|{key}"] = {"delta": d, "ktc_lo": lo, "ktc_hi": hi}
        _kl = ("⚠ chưa tách được khỏi nhiễu" if lo <= 0 <= hi
               else ("✅ có tác dụng" if d > 0 else "❌ làm tụt"))
        print(f"{TEN[fac]:<14}{len(cap):>8}{d:>+10.4f}   [{lo:+.4f}, {hi:+.4f}]   {_kl}")
        if len(cap) < 2:
            print(f"{'':<14}⚠ chỉ {len(cap)}/2 cặp — chưa phải tác động chính đầy đủ")

print(f"\n{'═'*92}\n  TƯƠNG TÁC\n{'═'*92}")
print("  Tương tác = (tác động của B khi A BẬT) − (khi A TẮT).")
print("  Âm = trùng nhau (A đã sửa sẵn thứ B định sửa). Dương = cộng hưởng.")

def tuong_tac(fac_a, fac_b, key):
    bat, tat = [], []
    for c, f, s, a, _ in MATRIX:
        spec = {"sft": f, "se": s, "ace": a}
        if spec[fac_b] or c not in RESULTS:
            continue
        want = dict(spec); want[fac_b] = True
        for c2, f2, s2, a2, _ in MATRIX:
            if {"sft": f2, "se": s2, "ace": a2} == want and c2 in RESULTS:
                (bat if spec[fac_a] else tat).append((_co(c, key), _co(c2, key)))
    return stats.ktc_tuong_tac(bat, tat), len(bat), len(tat)

for fa, fb in (("se", "ace"),):
    print(f"\n  {TEN[fa]} × {TEN[fb]}")
    for key, nhan in (("ea", "EA"), ("pa_strict", "PA")):
        r, n_bat, n_tat = tuong_tac(fa, fb, key)
        if r is None:
            print(f"    {nhan:<4} thiếu ô, không tính được")
            continue
        d, lo, hi = r
        _kl = ("⚠ chưa tách được khỏi nhiễu"
               if lo <= 0 <= hi else
               ("⚠ TRÙNG NHAU" if d < 0 else "✅ CỘNG HƯỞNG"))
        print(f"    {nhan:<4} {d:>+8.4f}   [{lo:+.4f}, {hi:+.4f}]   "
              f"({n_bat} cặp bật / {n_tat} cặp tắt)   {_kl}")

print(f"\n  ⚠ Tương tác cần rất nhiều mẫu mới tách được khỏi nhiễu. KTC chứa 0 nghĩa là")
print("     KHÔNG kết luận được gì — đừng viết 'cộng hưởng' hay 'trùng nhau' vào báo cáo.")


## §8. Tổ hợp tốt nhất có thật sự tốt hơn không

Chênh lệch trên bảng chưa đủ để kết luận. Phải kiểm định theo cặp, và phải so với **hai mốc**:

* ô `E+S` — đúng cấu hình cho kết quả tốt nhất ở mốc tham chiếu;
* ô tốt thứ nhì — để biết khoảng cách giữa hạng nhất và hạng nhì có ý nghĩa không.

In [ ]:
print(f"\n{'═'*88}\n  KIỂM ĐỊNH (McNemar theo cặp, n={len(test_all)})\n{'═'*88}")
comparisons = []
_ranked = sorted(_order, key=lambda c: -RESULTS[c][1]["EA"])
_best, _second = _ranked[0], (_ranked[1] if len(_ranked) > 1 else None)

_todo = []
if "E+S" in RESULTS and _best != "E+S":
    _todo.append(("E+S", _best, "Tổ hợp tốt nhất so với cấu hình tham chiếu (E+S)"))
if _second:
    _todo.append((_second, _best, "Hạng nhất so với hạng nhì"))
if "E+S" in RESULTS and "E+A" in RESULTS:
    _todo.append(("E+S", "E+A", "ACE có thay được self-eval không (rẻ hơn 1 lượt sinh)"))

for a, b, label in _todo:
    for key in ("ea", "pa_strict"):
        comparisons.append(stats.compare_pair(
            RESULTS[a][0], RESULTS[b][0], key=key, label=label,
            name_base=NICE[a], name_variant=NICE[b]))

In [ ]:
print(f"\n{'═'*88}\n  CHI PHÍ TRÊN MỖI ĐIỂM EA\n{'═'*88}")
_ref = "E"
if _ref in RESULTS:
    print(f"  Mốc: ô {_ref} ({NICE[_ref]}) — EA {ea(_ref):.4f}, 1 lượt sinh/mẫu, "
          f"không huấn luyện")
    print(f"\n{'ô':<8}{'ΔEA vs E':>10}{'lượt sinh':>11}{'huấn luyện':>12}"
          f"{'ΔEA / lượt sinh thêm':>22}")
    for c in _order:
        if c == _ref:
            continue
        f, s, a = _spec[c]
        gens = 2 if s else 1
        extra = gens - 1
        d = ea(c) - ea(_ref)
        per = f"{d/extra:+.4f}" if extra else "— (không tốn thêm)"
        print(f"{c:<8}{d:>+10.4f}{gens:>11}{'có' if f else 'không':>12}{per:>22}")
    print(f"\n  Cột cuối cho biết mỗi lượt sinh thêm mua được bao nhiêu EA.")
    print(f"  Ô không tốn thêm lượt sinh (ACE không kèm self-eval) mà vẫn dương là lựa chọn")
    print(f"  rẻ nhất — đáng nêu riêng trong bài khi nói về bối cảnh tài nguyên hạn chế.")

## §9. Lưu kết quả ma trận

In [ ]:
# Hai ô MỤC TIÊU (hậu tố *) KHÔNG nằm trong MATRIX nên không có trong `_order`.
# Phải thêm tay vào CSV — nếu quên thì đúng hai ô trả lời mục tiêu 70/70 lại vắng mặt
# khỏi file kết quả, mà đó mới là thứ đem đi báo cáo.
_xuat = list(_order) + [c for c in ("E+S*",) if c in RESULTS]

_csv = os.path.join(OUTPUT_DIR, f"ma_tran_to_hop_{STAMP}.csv")
with open(_csv, "w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["o", "cau_hinh", "sft", "self_eval", "ace",
                                      "cau_hinh_sinh", "EA", "PA_strict", "PA_loose",
                                      "luot_sinh", "phut"])
    w.writeheader()
    for c in _xuat:
        _goc = c.rstrip("*")
        sf, se, ac = _spec[_goc]
        m = RESULTS[c][1]
        _sinh = ("K mẫu + ví dụ động" if c.endswith("*") else "1 mẫu, temp 0.1")
        w.writerow({"o": c, "cau_hinh": NICE[_goc], "sft": int(sf), "self_eval": int(se),
                    "ace": int(ac), "cau_hinh_sinh": _sinh,
                    "EA": round(m["EA"]*100, 2),
                    "PA_strict": round(m["PA_strict"]*100, 2),
                    "PA_loose": round(m["PA_loose"]*100, 2),
                    "luot_sinh": 2 if se else 1, "phut": m.get("minutes", "")})

with open(os.path.join(OUTPUT_DIR, f"tuong_tac_{STAMP}.json"), "w", encoding="utf-8") as f:
    json.dump({"main_effects": MAIN,
               "matrix": {c: RESULTS[c][1] for c in _order},
               "o_muc_tieu": {c: RESULTS[c][1] for c in _xuat if c.endswith("*")},
               "comparisons": comparisons, "best": _best},
              f, ensure_ascii=False, indent=1, default=str)
print(f"[GHI] {_csv}  ({len(_xuat)} ô"
      + (f", gồm {sum(1 for c in _xuat if c.endswith(chr(42)))} ô mục tiêu)"
         if any(c.endswith("*") for c in _xuat) else ")"))

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

vals = [ea(c) for c in _order]
cols = ["tab:green" if c == _best else "tab:blue" for c in _order]
bars = axes[0].bar(range(len(_order)), vals, color=cols)
for b, v in zip(bars, vals):
    axes[0].text(b.get_x()+b.get_width()/2, v+0.008, f"{v:.3f}", ha="center", fontsize=8)
axes[0].set_xticks(range(len(_order)))
axes[0].set_xticklabels(_order, fontsize=9)
axes[0].set_ylim(0, 1); axes[0].grid(axis="y", alpha=0.3)
axes[0].set_title(f"Ma trận tổ hợp — EA ({len(test_all)} mẫu)")

# MAIN có khoá "<kỹ thuật>|<chỉ số>" và giá trị dict(delta, ktc_lo, ktc_hi).
# Chỉ vẽ EA, và vẽ kèm thanh sai số — cột trần không có KTC là cột dễ đọc sai nhất.
nm = {"se": "Self-eval", "ace": "ACE"}
_ea_keys = [k for k in MAIN if k.endswith("|ea")]
if _ea_keys:
    ten = [nm[k.split("|")[0]] for k in _ea_keys]
    vs = [MAIN[k]["delta"] for k in _ea_keys]
    err = [[v - MAIN[k]["ktc_lo"] for k, v in zip(_ea_keys, vs)],
           [MAIN[k]["ktc_hi"] - v for k, v in zip(_ea_keys, vs)]]
    # KTC chứa 0 → tô xám, đừng tô xanh/đỏ cho một con số chưa tách được khỏi nhiễu
    cols = ["tab:gray" if MAIN[k]["ktc_lo"] <= 0 <= MAIN[k]["ktc_hi"]
            else ("tab:green" if MAIN[k]["delta"] > 0 else "tab:red") for k in _ea_keys]
    bars = axes[1].bar(ten, vs, color=cols, yerr=err, capsize=4)
    for b, v in zip(bars, vs):
        axes[1].text(b.get_x()+b.get_width()/2, v, f"{v:+.3f}", ha="center",
                     va="bottom" if v >= 0 else "top", fontsize=9)
    axes[1].axhline(0, color="black", lw=0.8)
    axes[1].set_title("Tác động chính (xám = KTC chứa 0)")
axes[1].set_ylabel("ΔEA trung bình"); axes[1].grid(axis="y", alpha=0.3)
axes[1].set_ylabel("ΔEA trung bình")

# ACE giúp bao nhiêu, tuỳ có self-eval hay không
_g = {"chưa self-eval": [], "đã self-eval": []}
for c, f, s, a, _ in MATRIX:
    if a:
        continue
    want = {"sft": f, "se": s, "ace": True}
    for c2, f2, s2, a2, _ in MATRIX:
        if {"sft": f2, "se": s2, "ace": a2} == want and ea(c) is not None and ea(c2) is not None:
            _g["đã self-eval" if s else "chưa self-eval"].append(ea(c2) - ea(c))
_lbl = [k for k in _g if _g[k]]
if _lbl:
    axes[2].bar(_lbl, [sum(_g[k])/len(_g[k]) for k in _lbl],
                color=["tab:blue", "tab:orange"][:len(_lbl)])
    for i, k in enumerate(_lbl):
        v = sum(_g[k])/len(_g[k])
        axes[2].text(i, v, f"{v:+.3f}", ha="center",
                     va="bottom" if v >= 0 else "top", fontsize=9)
axes[2].axhline(0, color="black", lw=0.8)
axes[2].set_ylabel("ΔEA do ACE mang lại"); axes[2].grid(axis="y", alpha=0.3)
axes[2].set_title("ACE giúp nhiều hơn hay ít hơn sau khi self-eval?")

plt.tight_layout()
_png = os.path.join(OUTPUT_DIR, f"ma_tran_{STAMP}.png")
plt.savefig(_png, dpi=150); plt.show()
print(f"[GHI] {_png}")

## Cách diễn giải

| Quan sát | Kết luận | Viết gì trong bài |
|---|---|---|
| Tương tác SFT×ACE **âm rõ** | Hai cái học cùng thứ từ train | Chọn một: ACE nếu muốn khỏi huấn luyện, SFT nếu muốn khỏi tốn ngữ cảnh |
| Tương tác SFT×ACE **≈ 0** | Nhét vào trọng số và nhét vào ngữ cảnh là hai kênh độc lập | Kết quả đáng giá: dùng cả hai cộng dồn được |
| Tương tác SE×ACE **âm** | Hai cơ chế sửa lỗi chồng lấn | Bỏ self-eval, dùng ACE → rẻ hơn một nửa |
| `E+A` ≥ `E+S` | **ACE thay được self-eval** với nửa chi phí | Đóng góp mạnh cho luận điểm tài nguyên hạn chế |
| `F+S+A` tốt nhất, p < 0.05 vs `E+S` | Tổ hợp đầy đủ thắng cấu hình tham chiếu | Con số chính của báo cáo |

**Cạm bẫy khi đọc:** ma trận này sinh ra nhiều so sánh, nên dễ có cái "đạt p < 0.05" do may
mắn. Kết luận mạnh chỉ nên dựa vào hai so sánh đã định trước — tổ hợp tốt nhất so với `E+S`,
và `E+A` so với `E+S` — phần còn lại xem như thăm dò.

**Tiếp theo:** `07_final_report.ipynb` gộp cả chuỗi 5 nấc lẫn ma trận này thành báo cáo cuối.